# Chapter 6 — Deep Learning for Sentiment and Student Well-Being Analysis

**Companion notebook | Text analysis, BERT fine-tuning, and interpretation**

This notebook follows the technical sequence of Chapter 6: linguistic exploration, tokenisation, class-weighted BERT classification, error analysis, educational keywords, thematic modelling, and cautious interpretation.

**Dataset:** Suchintika Sarkar, [*Sentiment Analysis for Mental Health*](https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health) (Kaggle). Download **`Combined Data.csv`** and place it next to this notebook or in the `data/` directory.

**Scope:** The dataset contains labelled mental-health-related text from online sources. It is **not a verified sample of school students**, and its labels must **not** be interpreted as clinical diagnoses or used for automatic student triage. The figures reflect the dataset and methods executed here, not guaranteed reproductions of historical published results.

**Execution modes:** Descriptive analyses run without downloading model weights. Set `RUN_TOKEN_ANALYSIS=True` to compute actual BERT token lengths, and set `RUN_TRAINING=True` to fine-tune the classifier and produce test-set evaluation charts. Model training requires installed dependencies, downloaded BERT weights, and substantial computing resources.

**Dependencies:** `pandas numpy matplotlib seaborn scikit-learn torch transformers jupyter`. Install in your environment using `python -m pip install pandas numpy matplotlib seaborn scikit-learn torch transformers jupyter`.

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>01 | Environment and run settings</strong><br>
<span style="color:#405266;">Configure the data location, reproducible random seed, output directory, and optional model-training stages.</span>
</div>

In [ ]:
from pathlib import Path
import json, os, random, re, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_recall_fscore_support)

SEED = 42
RUN_TOKEN_ANALYSIS = False    # Requires access to the BERT tokenizer
RUN_TRAINING = False          # Set True for the complete fine-tuning experiment
MODEL_NAME = 'bert-base-uncased'
MAX_LENGTH = 128
EPOCHS = 3
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
MAX_TRAIN_SAMPLES = None       # None uses the complete training partition
MAX_TOKEN_SAMPLE = 5000       # Number of original texts used for the length diagnostic

random.seed(SEED)
np.random.seed(SEED)
OUT_DIR = Path('figures')
OUT_DIR.mkdir(exist_ok=True, parents=True)
sns.set_theme(style='whitegrid', context='notebook')

def save_figure(fig, filename):
    fig.tight_layout()
    for ext in ('pdf', 'png'):
        fig.savefig(OUT_DIR / f'{filename}.{ext}', dpi=220, bbox_inches='tight')
    plt.show()
    plt.close(fig)

print(f'Python {platform.python_version()} | Figures: {OUT_DIR.resolve()}')
print(f'Token analysis: {RUN_TOKEN_ANALYSIS} | Model training: {RUN_TRAINING}')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>02 | Load the source dataset</strong><br>
<span style="color:#405266;">Locate the downloaded CSV, normalise column names, and check the two required fields.</span>
</div>

In [ ]:
DATA_FILE = None  # Optional explicit path, e.g. Path('data/Combined Data.csv')
candidates = ([Path(DATA_FILE)] if DATA_FILE else []) + [
    Path('Combined Data.csv'), Path('data/Combined Data.csv'),
    Path('data/combined_data.csv'), Path('data.csv'), Path('data/data.csv')]
source_path = next((p for p in candidates if p.is_file()), None)
if source_path is None:
    raise FileNotFoundError(
        'Dataset not found. Download Combined Data.csv from '
        'https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health '
        'and place it in this notebook directory or the data/ directory.'
    )
raw = pd.read_csv(source_path, low_memory=False)
raw.columns = [str(c).strip().lower() for c in raw.columns]
raw = raw.loc[:, ~raw.columns.str.match(r'^unnamed')]
if not {'statement', 'status'}.issubset(raw.columns):
    raise ValueError(f'Expected statement and status columns; found {list(raw.columns)}')
print(f'Data source: {source_path} | Original rows: {len(raw):,}')
print(raw[['statement','status']].notna().sum().to_string())

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>03 | Prepare text and assess data quality</strong><br>
<span style="color:#405266;">Remove missing or empty examples, inspect class frequencies, and record repeated or conflicting statements.</span>
</div>

In [ ]:
df = raw[['statement','status']].copy().dropna(subset=['statement','status'])
df['statement'] = df['statement'].astype(str).str.strip()
df['status'] = df['status'].astype(str).str.strip()
df = df.loc[df['statement'].ne('') & df['status'].ne('')].copy()
df['text_key'] = df['statement'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
conflicting = df.groupby('text_key')['status'].nunique()
conflicting_keys = set(conflicting[conflicting.gt(1)].index)
print(f'Usable rows: {len(df):,}')
print(f'Exact repeated text records: {int(df["text_key"].duplicated().sum()):,}')
print(f'Texts associated with conflicting labels: {len(conflicting_keys):,}')
print('Label counts:')
print(df['status'].value_counts().to_string())

# Keep original examples for descriptive visualisations; use distinct texts for training.
model_df = df.loc[~df['text_key'].isin(conflicting_keys)].drop_duplicates('text_key').copy()
if model_df['status'].nunique() < 2:
    raise ValueError('At least two distinct label categories are required.')
print(f'Unique, non-conflicting texts available for modelling: {len(model_df):,}')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>04 | Figure 6.2 — Label distribution</strong><br>
<span style="color:#405266;">Display the number of original text records in each annotation category.</span>
</div>

In [ ]:
counts = df['status'].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(counts.index, counts.values, color='#57768b')
ax.set(title='Distribution of labelled text categories', xlabel='Source label', ylabel='Number of statements')
ax.tick_params(axis='x', rotation=22)
for i, n in enumerate(counts.values): ax.text(i, n, f'{n:,}', ha='center', va='bottom', fontsize=9)
save_figure(fig, 'Figure_6_02_status_distribution')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>05 | Linguistic features</strong><br>
<span style="color:#405266;">Measure word count, mean word length, and lexical diversity consistently from each statement.</span>
</div>

In [ ]:
word_re = re.compile(r"\b[\w']+\b", flags=re.UNICODE)
def linguistic_features(text):
    words = word_re.findall(text.lower())
    count = len(words)
    return pd.Series({
        'word_count': count,
        'avg_word_len': (sum(len(w) for w in words) / count if count else 0.0),
        'lexical_diversity': (len(set(words)) / count if count else 0.0)
    })
features = df['statement'].apply(linguistic_features)
df = pd.concat([df, features], axis=1)
print(df.groupby('status')[['word_count','avg_word_len','lexical_diversity']].median().round(3))

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>06 | Figure 6.4 — Linguistic volume</strong><br>
<span style="color:#405266;">Compare word-count distributions across labels without drawing clinical conclusions.</span>
</div>

In [ ]:
order = df['status'].value_counts().index.tolist()
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.boxplot(data=df, x='status', y='word_count', order=order, showfliers=False, ax=ax, color='#8ab1bc')
ax.set(title='Linguistic volume by source label', xlabel='Source label', ylabel='Words per statement')
ax.tick_params(axis='x', rotation=25)
save_figure(fig, 'Figure_6_04_word_count')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>07 | Figure 6.5 — Word length</strong><br>
<span style="color:#405266;">Compare the descriptive mean word length; this is not a validated readability score.</span>
</div>

In [ ]:
summary = df.groupby('status')['avg_word_len'].mean().reindex(order)
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(summary.index, summary.values, color='#67969a')
ax.set(title='Mean word length across source labels', xlabel='Source label', ylabel='Characters per word')
ax.tick_params(axis='x', rotation=25)
save_figure(fig, 'Figure_6_05_word_length')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>08 | Figure 6.6 — Lexical diversity</strong><br>
<span style="color:#405266;">Visualise the type–token ratio of each statement and compare its distribution by label.</span>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.violinplot(data=df, x='status', y='lexical_diversity', order=order,
               inner='quartile', cut=0, color='#a9a0bc', ax=ax)
ax.set(title='Lexical diversity of statements', xlabel='Source label', ylabel='Unique-token ratio')
ax.tick_params(axis='x', rotation=25)
save_figure(fig, 'Figure_6_06_lexical_diversity')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>09 | Partition data and encode labels</strong><br>
<span style="color:#405266;">Create stratified train, validation, and test partitions, avoiding identical texts across partitions.</span>
</div>

In [ ]:
labels = LabelEncoder()
model_df['label'] = labels.fit_transform(model_df['status'])
class_names = list(labels.classes_)
if model_df['label'].value_counts().min() < 7:
    raise ValueError('Each class needs at least seven distinct texts for the stratified partitions.')
train_val, test_df = train_test_split(model_df, test_size=0.15, random_state=SEED,
                                      stratify=model_df['label'])
train_df, val_df = train_test_split(train_val, test_size=0.15/0.85, random_state=SEED,
                                    stratify=train_val['label'])
if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(train_df):
    train_df, _ = train_test_split(train_df, train_size=MAX_TRAIN_SAMPLES,
                                  random_state=SEED, stratify=train_df['label'])
assert not (set(train_df['text_key']) & set(val_df['text_key']))
assert not (set(train_df['text_key']) & set(test_df['text_key']))
assert not (set(val_df['text_key']) & set(test_df['text_key']))
print('Class mapping:', dict(enumerate(class_names)))
print('Train / Validation / Test:', len(train_df), len(val_df), len(test_df))

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>10 | Figure 6.8 — Training class weights</strong><br>
<span style="color:#405266;">Compute inverse-frequency weights from the training partition only.</span>
</div>

In [ ]:
classes = np.arange(len(class_names))
weight_values = compute_class_weight(class_weight='balanced', classes=classes,
                                     y=train_df['label'].to_numpy())
weights = dict(zip(class_names, weight_values))
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(list(weights), list(weights.values()), color='#96758a')
ax.set(title='Class weights estimated from training data', xlabel='Source label', ylabel='Weight')
ax.tick_params(axis='x', rotation=25)
save_figure(fig, 'Figure_6_08_training_class_weights')
print(pd.Series(weights, name='training_weight').round(3))

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>11 | Figure 6.7 — BERT token-length diagnostic</strong><br>
<span style="color:#405266;">Optionally measure uncapped WordPiece sequence lengths on a fixed sample; compare them with the 128-token limit.</span>
</div>

In [ ]:
if RUN_TOKEN_ANALYSIS or RUN_TRAINING:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    texts = df['statement'].sample(min(MAX_TOKEN_SAMPLE, len(df)), random_state=SEED).tolist()
    token_lengths = [len(v) for v in tokenizer(texts, add_special_tokens=True,
                         truncation=False, padding=False)['input_ids']]
    fraction_truncated = np.mean(np.array(token_lengths) > MAX_LENGTH)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(token_lengths, bins=45, density=True, color='#7c789b', alpha=0.85)
    ax.axvline(MAX_LENGTH, linestyle='--', color='#b34747',
               label=f'{MAX_LENGTH}-token limit')
    ax.set(title='BERT token lengths (sample of source texts)',
           xlabel='Tokens including special tokens', ylabel='Density')
    ax.legend()
    save_figure(fig, 'Figure_6_07_bert_token_lengths')
    print(f'Sampled texts: {len(texts):,} | Exceeding {MAX_LENGTH} tokens: {fraction_truncated:.1%}')
else:
    print('Token-length analysis skipped. Set RUN_TOKEN_ANALYSIS=True after installing transformers.')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>12 | Prepare BERT data loaders</strong><br>
<span style="color:#405266;">When training is enabled, tokenise each data partition to 128 tokens and construct batches for optimisation.</span>
</div>

In [ ]:
if RUN_TRAINING:
    import torch
    from torch.utils.data import Dataset, DataLoader
    from transformers import BertForSequenceClassification
    import transformers
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Training device:', device)

    class TextDataset(Dataset):
        def __init__(self, frame):
            self.text = frame['statement'].tolist()
            self.y = frame['label'].astype(int).tolist()
        def __len__(self): return len(self.y)
        def __getitem__(self, idx):
            tokens = tokenizer(self.text[idx], max_length=MAX_LENGTH,
                               truncation=True, padding='max_length', return_tensors='pt')
            return {**{k: v.squeeze(0) for k,v in tokens.items()},
                    'labels': torch.tensor(self.y[idx], dtype=torch.long)}

    train_loader = DataLoader(TextDataset(train_df), batch_size=TRAIN_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TextDataset(val_df), batch_size=EVAL_BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(TextDataset(test_df), batch_size=EVAL_BATCH_SIZE, shuffle=False)
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(class_names))
    model.to(device)
else:
    print('Data loaders and model initialisation skipped; enable RUN_TRAINING for the full experiment.')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>13 | Class-weighted training procedure</strong><br>
<span style="color:#405266;">Fine-tune BERT with AdamW, class-weighted cross-entropy, and validation-based checkpoint selection.</span>
</div>

In [ ]:
if RUN_TRAINING:
    from torch.optim import AdamW
    from transformers import get_linear_schedule_with_warmup
    from sklearn.metrics import f1_score

    criterion = torch.nn.CrossEntropyLoss(
        weight=torch.as_tensor(weight_values, dtype=torch.float, device=device))
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    steps = max(1, EPOCHS * len(train_loader))
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0,
                                                 num_training_steps=steps)

    def run_epoch(loader, train):
        model.train(train)
        total_loss, n = 0.0, 0
        true_labels, predictions = [], []
        for batch in loader:
            y = batch.pop('labels').to(device)
            x = {k: v.to(device) for k, v in batch.items()}
            with torch.set_grad_enabled(train):
                logits = model(**x).logits
                loss = criterion(logits, y)
                if train:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
            total_loss += float(loss.item()) * len(y)
            n += len(y)
            true_labels.extend(y.detach().cpu().tolist())
            predictions.extend(logits.argmax(dim=-1).detach().cpu().tolist())
        return {'loss': total_loss/n, 'accuracy': accuracy_score(true_labels, predictions),
                'macro_f1': f1_score(true_labels, predictions, average='macro', zero_division=0)}

    history = []
    best_f1, best_epoch = -1, None
    best_dir = Path('model_checkpoint')
    for epoch in range(1, EPOCHS+1):
        train_stats = run_epoch(train_loader, train=True)
        val_stats = run_epoch(val_loader, train=False)
        history.append({'epoch': epoch, **{f'train_{k}':v for k,v in train_stats.items()},
                        **{f'val_{k}':v for k,v in val_stats.items()}})
        print(f'Epoch {epoch}: train loss={train_stats["loss"]:.3f}, '
              f'val accuracy={val_stats["accuracy"]:.3f}, val macro-F1={val_stats["macro_f1"]:.3f}')
        if val_stats['macro_f1'] > best_f1:
            best_f1, best_epoch = val_stats['macro_f1'], epoch
            best_dir.mkdir(exist_ok=True)
            model.save_pretrained(best_dir)
            tokenizer.save_pretrained(best_dir)
    pd.DataFrame(history).to_csv('training_history.csv', index=False)
    print(f'Best validation macro-F1: {best_f1:.3f} (epoch {best_epoch})')
else:
    print('Training skipped; historical training curves will not be substituted.')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>14 | Figure 6.9 — Recorded training dynamics</strong><br>
<span style="color:#405266;">Plot the measured accuracy and cross-entropy loss for training and validation; no values are entered manually.</span>
</div>

In [ ]:
if RUN_TRAINING:
    hist = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for split in ('train','val'):
        axes[0].plot(hist.epoch, hist[f'{split}_accuracy'], marker='o', label=split.title())
        axes[1].plot(hist.epoch, hist[f'{split}_loss'], marker='o', label=split.title())
    axes[0].set(xlabel='Epoch', ylabel='Accuracy', title='(a) Classification accuracy')
    axes[1].set(xlabel='Epoch', ylabel='Cross-entropy loss', title='(b) Optimisation loss')
    for ax in axes: ax.legend(); ax.set_xticks(hist.epoch)
    save_figure(fig, 'Figure_6_09_training_dynamics')
else:
    print('Figure 6.9 requires a completed training run.')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>15 | Figure 6.10 — Held-out evaluation</strong><br>
<span style="color:#405266;">Evaluate the best validation checkpoint once on the untouched test partition and calculate its confusion matrix.</span>
</div>

In [ ]:
if RUN_TRAINING:
    model = BertForSequenceClassification.from_pretrained(best_dir).to(device)
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in test_loader:
            y_true.extend(batch.pop('labels').tolist())
            scores = model(**{k:v.to(device) for k,v in batch.items()}).logits
            y_pred.extend(scores.argmax(dim=-1).cpu().tolist())
    print(classification_report(y_true, y_pred, labels=classes, target_names=class_names,
                                zero_division=0, digits=3))
    matrix = confusion_matrix(y_true, y_pred, labels=classes)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set(title='Test-set confusion matrix', xlabel='Predicted source label', ylabel='Observed source label')
    ax.tick_params(axis='x', rotation=35)
    save_figure(fig, 'Figure_6_10_test_confusion_matrix')
    pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv('test_confusion_matrix.csv')
else:
    print('Figure 6.10 requires trained predictions; no stored counts are substituted.')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>16 | Figure 6.11 — Linguistic-length distribution</strong><br>
<span style="color:#405266;">Compare word-length distributions across source labels without equating word length with psychological status.</span>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
plot_df = df.loc[df['avg_word_len'].between(1, 15)].copy()
for category in order:
    values = plot_df.loc[plot_df.status.eq(category), 'avg_word_len']
    if len(values) >= 5 and values.nunique() > 1:
        ax.hist(values.to_numpy(dtype=float), bins=25, density=True, histtype='step', linewidth=1.6, label=category)
ax.set(title='Distribution of mean word length', xlabel='Characters per word', ylabel='Density')
ax.legend(loc='best', fontsize=8)
save_figure(fig, 'Figure_6_11_mean_word_length_distribution')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>17 | Figure 6.12 — Educational vocabulary</strong><br>
<span style="color:#405266;">Calculate the proportion of statements containing selected education-related terms within each label.</span>
</div>

In [ ]:
keywords = ['exam','grade','homework','school','class','teacher','study',
            'university','college','assignment']
keyword_frame = pd.DataFrame(index=df.index)
keyword_frame['status'] = df['status']
for keyword in keywords:
    keyword_frame[keyword] = df['statement'].str.contains(
        rf'\b{re.escape(keyword)}\b', case=False, na=False, regex=True)
keyword_rates = keyword_frame.groupby('status')[keywords].mean().reindex(order)
fig, ax = plt.subplots(figsize=(12, 5.6))
sns.heatmap(keyword_rates, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax)
ax.set(title='Educational vocabulary: fraction of statements containing each term',
       xlabel='Keyword', ylabel='Source label')
save_figure(fig, 'Figure_6_12_education_keyword_associations')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>18 | Figure 6.13 — Latent themes in labelled text</strong><br>
<span style="color:#405266;">Fit three exploratory LDA topics to a sample of non-Normal records, then display their leading terms.</span>
</div>

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
text_subset = df.loc[df['status'].str.casefold().ne('normal'), 'statement']
if len(text_subset) < 60:
    text_subset = df['statement']
text_subset = text_subset.sample(min(len(text_subset), 10000), random_state=SEED)
vectorizer = CountVectorizer(stop_words='english', min_df=2, max_df=0.90, max_features=2500)
term_matrix = vectorizer.fit_transform(text_subset)
if term_matrix.shape[1] < 9:
    raise ValueError('Not enough distinct vocabulary remains for a three-topic analysis.')
lda = LatentDirichletAllocation(n_components=3, learning_method='batch', max_iter=15,
                                random_state=SEED)
lda.fit(term_matrix)
feature_names = vectorizer.get_feature_names_out()
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for i, (topic, ax) in enumerate(zip(lda.components_, axes)):
    ix = np.argsort(topic)[-10:][::-1]
    ax.barh(feature_names[ix][::-1], topic[ix][::-1], color='#608c91')
    ax.set(title=f'Topic {i+1}', xlabel='Term weight')
fig.suptitle('Exploratory thematic structure of labelled text', fontsize=13)
save_figure(fig, 'Figure_6_13_exploratory_topics')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>19 | Figure 6.14 — Illustrative review categories</strong><br>
<span style="color:#405266;">Group existing labels into descriptive review groups for a teaching example; these are not validated referral decisions.</span>
</div>

In [ ]:
# Conceptual grouping only. Do not use individual predictions for clinical decisions.
illustrative_groups = {
    'Normal': 'Routine contextual review',
    'Anxiety': 'Additional contextual review', 'Stress': 'Additional contextual review',
    'Depression': 'Additional contextual review', 'Bipolar': 'Additional contextual review',
    'Personality disorder': 'Additional contextual review',
    'Suicidal': 'Prompt professional assessment'
}
missing_groups = sorted(set(df['status']) - set(illustrative_groups))
if missing_groups:
    print('Unmapped labels grouped as Other:', missing_groups)
review_labels = df['status'].map(illustrative_groups).fillna('Other')
review_counts = review_labels.value_counts()
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(review_counts.index[::-1], review_counts.values[::-1], color='#73909d')
ax.set(title='Illustrative label grouping (not an intervention rule)',
       xlabel='Number of source statements', ylabel='')
save_figure(fig, 'Figure_6_14_illustrative_review_groups')
print('These proportions describe online statements, not students requiring care.')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>20 | Figure 6.15 — Length-based diagnostic</strong><br>
<span style="color:#405266;">Inspect differences in statement length across labels; this descriptive comparison alone cannot establish model fairness.</span>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.5))
for label in order:
    values = df.loc[df.status.eq(label), 'word_count'].clip(upper=250)
    if len(values) >= 5 and values.nunique() > 1:
        ax.hist(values.to_numpy(dtype=float), bins=35, density=True, histtype='step', linewidth=1.5, label=label)
ax.set(title='Statement-length distribution across source labels',
       xlabel='Word count (display capped at 250)', ylabel='Density')
ax.legend(fontsize=8)
save_figure(fig, 'Figure_6_15_linguistic_length_diagnostic')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>21 | Optional length-stratified error check</strong><br>
<span style="color:#405266;">If test predictions exist, compare classification recall across statement-length groups.</span>
</div>

In [ ]:
if RUN_TRAINING:
    audit = test_df[['statement','label']].reset_index(drop=True).copy()
    audit['word_count'] = audit['statement'].str.split().str.len()
    audit['prediction'] = y_pred
    audit['length_group'] = pd.qcut(audit['word_count'].rank(method='first'), q=3,
                                    labels=['Short','Medium','Long'])
    results = []
    for group, part in audit.groupby('length_group', observed=True):
        score = precision_recall_fscore_support(part['label'], part['prediction'],
                         labels=classes, average='macro', zero_division=0)
        results.append({'length_group':str(group), 'n':len(part), 'macro_precision':score[0],
                        'macro_recall':score[1], 'macro_f1':score[2]})
    audit_summary = pd.DataFrame(results)
    print(audit_summary.round(3).to_string(index=False))
    audit_summary.to_csv('test_error_by_length.csv', index=False)
else:
    print('Length-stratified error analysis requires held-out predictions.')

<div style="border-left:4px solid #4b647c; background:#f4f7fa; padding:10px 14px; border-radius:7px; margin:14px 0 8px 0;">
<strong>22 | Outputs and reproducibility record</strong><br>
<span style="color:#405266;">List the figures generated in this run and record parameters without substituting unpublished results.</span>
</div>

In [ ]:
records = {
    'source': 'https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health',
    'local_file': str(source_path), 'clean_rows': int(len(df)),
    'unique_modelling_rows': int(len(model_df)),
    'seed': SEED, 'model': MODEL_NAME, 'maximum_sequence_length': MAX_LENGTH,
    'train_epochs': EPOCHS, 'learning_rate': LEARNING_RATE,
    'ran_token_analysis': RUN_TOKEN_ANALYSIS or RUN_TRAINING,
    'ran_training': RUN_TRAINING,
    'figures': sorted(p.name for p in OUT_DIR.glob('*.pdf')),
    'note': 'Exploratory analysis of source text. No clinical or student-level inference.'
}
Path('chapter06_run_manifest.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
print(f'Figures generated: {len(records["figures"])}')
for figure in records['figures']:
    print(' •', figure)
print('Manifest:', Path('chapter06_run_manifest.json').resolve())

### Notes on interpretation

- **Figure 6.1:** The chapter's conceptual framework is a separate illustration rather than an output of data analysis.
- **Figure 6.3:** The chapter refers to a relationship between earlier academic assessments and risk. The supplied text dataset contains no verified paired academic outcomes, so this figure cannot be reproduced from this source.
- **Figures 6.9–6.10:** These are produced only after actual model training and evaluation. The original notebook contained manually entered numbers and a stored confusion matrix; these are **not** reused as newly computed findings.
- **Figure 6.14:** The review categories are illustrative groupings, not validated psychological screening, counselling recommendations, or emergency-response decisions.
- **Fairness:** Similarity in text-length distributions does not establish fairness. Any institutional use requires privacy review, external validation, qualified oversight, and separate assessment of subgroup errors.

Running this notebook with the same dataset and the same settings may still yield results different from the book's historical figures because the original notebook did not contain a complete recorded training protocol, checkpoint, or model outputs.